In [ ]:
import geopandas as gpd

# 路径可能需要根据您的具体情况进行调整
file_path = r'G:\Hangkai\CONUS Forest Edge Mapping\ecoregion\2001_ecoregion.shp'
data_2001 = gpd.read_file(file_path)

In [ ]:
data_2001

In [ ]:
print(data_2001.head())


In [ ]:
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
summary_data = []  # 使用列表来收集所有行

# 假设您有从2001到2020年的shapefiles
for year in tqdm(range(2019, 2020)):
    file_path = f'G:/Hangkai/CONUS Forest Edge Mapping/ecoregion/{year}_ecoregion.shp'
    data = gpd.read_file(file_path)
    
    # 假设REGION或者其他字段标识了ecoregion
    for region in data['NA_L1NAME'].unique():
        region_data = data[data['NA_L1NAME'] == region]
        
        # 计算每种类型的edge pixel数量
        for histo_col in [f'HISTO_{i}' for i in range(1, 16)]:
            total_edges = region_data[histo_col].sum()
            
            # 收集数据
            summary_data.append({
                'Year': year,
                'Ecoregion': region,
                'EdgeType': histo_col,
                'TotalEdges': total_edges
            })

# 将收集到的数据转换为DataFrame
summary_df = pd.DataFrame(summary_data)

# 现在您有了一个包含每一年、每个ecoregion及其edge类型总数的汇总数据的DataFrame
print(summary_df.head())


In [ ]:
summary_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 边缘类型分布示例
ecoregion_example = 'MARINE WEST COAST FOREST'  # 示例ecoregion，您可以选择任意一个
ecoregion_data = summary_df[summary_df['Ecoregion'] == ecoregion_example]

plt.figure(figsize=(10, 6))
sns.barplot(data=ecoregion_data, x='EdgeType', y='TotalEdges', hue='Year')
plt.title(f'Edge Type Distribution in {ecoregion_example}')
plt.xlabel('Edge Type')
plt.ylabel('Total Edge Pixels')
plt.xticks(rotation=45)
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# 年度对比示例
year_example = 2019  # 示例年份，您可以选择任意一个
year_data = summary_df[summary_df['Year'] == year_example]

plt.figure(figsize=(10, 6))
sns.barplot(data=year_data, x='Ecoregion', y='TotalEdges', hue='EdgeType')
plt.title(f'Yearly Comparison of Edge Pixels in {year_example}')
plt.xlabel('Ecoregion')
plt.ylabel('Total Edge Pixels')
plt.xticks(rotation=45)
plt.legend(title='Edge Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# 假设summary_df已经有了正确的数据结构
# 先创建一个新的DataFrame来存储每年每个ecoregion的边缘统计
edges_analysis_df = []

# 分组处理以计算每个ecoregion每年的边缘统计
for (year, ecoregion), group in summary_df.groupby(['Year', 'Ecoregion']):
    edge_counts = group.set_index('EdgeType')['TotalEdges'].to_dict()
    
    # 使用HISTO_i来获取对应的edge_counts值
    edges_individual = {
        'North': sum([edge_counts.get(f'HISTO_{i}', 0) for i in [8, 12, 10, 9, 14, 13, 11, 15]]),
        'South': sum([edge_counts.get(f'HISTO_{i}', 0) for i in [4, 12, 6, 5, 14, 13, 7, 15]]),
        'West':  sum([edge_counts.get(f'HISTO_{i}', 0) for i in [2, 10, 6, 3, 14, 11, 7, 15]]),
        'East':  sum([edge_counts.get(f'HISTO_{i}', 0) for i in [1, 9, 5, 3, 13, 11, 7, 15]])
    }
    
    # 组合边界统计
    edge_pixel_combined = {
        '1 Edge': sum([edge_counts.get(f'HISTO_{i}', 0) for i in [1, 2, 4, 8]]),
        '2 Edges': sum([edge_counts.get(f'HISTO_{i}', 0) for i in [3, 5, 6, 9, 10, 12]]),
        '3 Edges': sum([edge_counts.get(f'HISTO_{i}', 0) for i in [7, 11, 13, 14]]),
        '4 Edges': edge_counts.get(f'HISTO_{15}', 0)
    }
    
    # 将结果添加到列表中
    edges_analysis_df.append({
        'Year': year,
        'Ecoregion': ecoregion,
        **edges_individual,
        **edge_pixel_combined
    })

# 将列表转换成DataFrame
edges_analysis_df = pd.DataFrame(edges_analysis_df)

# 现在edges_analysis_df包含每年每个ecoregion的不同方向和边数的edge pixels统计
print(edges_analysis_df.head())

In [ ]:
import pandas as pd

# I assume edges_analysis_df has already been defined and created as shown in your script.
# Add a new column for total edge pixels by summing all edge types for each row
edges_analysis_df['Total Edge Pixels'] = edges_analysis_df['North'] + edges_analysis_df['South'] + edges_analysis_df['West'] + edges_analysis_df['East'] + edges_analysis_df['1 Edge'] + edges_analysis_df['2 Edges'] + edges_analysis_df['3 Edges'] + edges_analysis_df['4 Edges']

# Calculate total edge length in meters by multiplying the total pixels by 30
edges_analysis_df['Total Edge Length (m)'] = edges_analysis_df['Total Edge Pixels'] * 30

# Calculate the total edge length for the whole CONUS by summing all the edge lengths
total_edge_length_conus = edges_analysis_df['Total Edge Length (m)'].sum()

# Calculate the total edge length by ecoregion by summing the edge lengths within each ecoregion
total_edge_length_by_ecoregion = edges_analysis_df.groupby('Ecoregion')['Total Edge Length (m)'].sum()

# Printing results
print("Total Edge Length for the whole CONUS (km):", total_edge_length_conus/1000)
print("\nTotal Edge Length by Ecoregion (km):")
print(total_edge_length_by_ecoregion/1000)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Sample data, assuming you have these values in a dictionary or similar structure
data = {
    'Ecoregion': [
        'EASTERN TEMPERATE FORESTS', 'GREAT PLAINS', 'MARINE WEST COAST FOREST',
        'MEDITERRANEAN CALIFORNIA', 'NORTH AMERICAN DESERTS', 'NORTHERN FORESTS',
        'NORTHWESTERN FORESTED MOUNTAINS', 'SOUTHERN SEMIARID HIGHLANDS',
        'TEMPERATE SIERRAS', 'TROPICAL WET FORESTS', 'WATER'
    ],
    'Edge Length (km)': [
        22022271.60, 5813994.84, 580731.66, 736391.73, 2566728.39,
        2338695.15, 7163800.29, 102681.72, 902958.93, 227533.14, 8424.84
    ]
}

# Convert the dictionary to DataFrame
df = pd.DataFrame(data)

# Sorting the values for better visualization
df = df.sort_values(by='Edge Length (km)', ascending=False)

# Creating the bar chart
plt.figure(figsize=(14, 7))
plt.bar(df['Ecoregion'], df['Edge Length (km)'], color='cadetblue')
plt.title('Total Edge Length by Ecoregion (km)')
plt.xlabel('Ecoregion')
plt.ylabel('Edge Length (km)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()  # Adjust layout to make room for the rotated x-axis labels

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 计算边缘比例
edges_analysis_df['Total'] = edges_analysis_df['North'] + edges_analysis_df['South'] + edges_analysis_df['West'] + edges_analysis_df['East']
edges_analysis_df['North Proportion'] = edges_analysis_df['North'] / edges_analysis_df['Total']
edges_analysis_df['South Proportion'] = edges_analysis_df['South'] / edges_analysis_df['Total']
edges_analysis_df['West Proportion'] = edges_analysis_df['West'] / edges_analysis_df['Total']
edges_analysis_df['East Proportion'] = edges_analysis_df['East'] / edges_analysis_df['Total']

# 获取ecoregion列表
ecoregions = edges_analysis_df['Ecoregion'].unique()

# 设置图表大小
plt.figure(figsize=(20, 15))

# 对每个ecoregion绘制一个时间序列图
for i, ecoregion in enumerate(ecoregions, 1):
    plt.subplot(len(ecoregions), 1, i)
    ecoregion_data = edges_analysis_df[edges_analysis_df['Ecoregion'] == ecoregion]
    
    sns.lineplot(data=ecoregion_data, x='Year', y='North Proportion', label='North')
    sns.lineplot(data=ecoregion_data, x='Year', y='South Proportion', label='South')
    sns.lineplot(data=ecoregion_data, x='Year', y='West Proportion', label='West')
    sns.lineplot(data=ecoregion_data, x='Year', y='East Proportion', label='East')
    
    plt.title(f'Edge Direction Proportions Over Time in {ecoregion}')
    plt.xlabel('Year')
    plt.ylabel('Proportion')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
edges_analysis_df.sum()

In [ ]:
# 保存DataFrame为CSV文件
edges_analysis_df.to_csv('edges_analysis.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 计算南北方向和东西方向边缘的总比例
edges_analysis_df['North-South Proportion'] = edges_analysis_df['North Proportion'] + edges_analysis_df['South Proportion']
edges_analysis_df['East-West Proportion'] = edges_analysis_df['East Proportion'] + edges_analysis_df['West Proportion']

# 为每个生态区创建时间序列图，展示南北与东西方向的比例变化
ecoregions = edges_analysis_df['Ecoregion'].unique()
plt.figure(figsize=(20, 15))

for i, ecoregion in enumerate(ecoregions, 1):
    plt.subplot(len(ecoregions), 1, i)
    ecoregion_data = edges_analysis_df[edges_analysis_df['Ecoregion'] == ecoregion]
    
    sns.lineplot(data=ecoregion_data, x='Year', y='North-South Proportion', label='North-South')
    sns.lineplot(data=ecoregion_data, x='Year', y='East-West Proportion', label='East-West')
    
    plt.title(f'Edge Direction Proportions Over Time in {ecoregion}')
    plt.xlabel('Year')
    plt.ylabel('Proportion')
    plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 计算每个生态区每年的方向敏感性指标
edges_analysis_df['Directional Sensitivity'] = edges_analysis_df['North-South Proportion'] - edges_analysis_df['East-West Proportion']

# 汇总每个生态区的方向敏感性指标，这里我们取平均值作为表示
directional_sensitivity_avg = edges_analysis_df.groupby('Ecoregion')['Directional Sensitivity'].mean().reset_index()

# 对生态区按照方向敏感性指标排序
directional_sensitivity_avg = directional_sensitivity_avg.sort_values(by='Directional Sensitivity', ascending=False)
plot_directional_sensitivity_avg = directional_sensitivity_avg
plot_directional_sensitivity_avg["Directional Sensitivity"] = directional_sensitivity_avg["Directional Sensitivity"]*100
# 可视化不同生态区的方向敏感性指标
plt.figure(figsize=(12, 8))
sns.barplot(data=plot_directional_sensitivity_avg, x='Directional Sensitivity', y='Ecoregion', palette='coolwarm')
plt.xlabel('North & south edge propotion - East & west edge propotion (%)')
plt.ylabel('Ecoregion')
plt.show()



In [ ]:
plot_directional_sensitivity_avg

In [ ]:
edges_analysis_df

In [ ]:
import geopandas as gpd

# Path to the shapefiles
forest_depth_path = "G:/Hangkai/CONUS Forest Edge Mapping/2019/CONUS_2019_Forest_Depth.shp"
summed_edge_length_path = "G:/Hangkai/CONUS Forest Edge Mapping/2019/CONUS_2019_summed_edge_length.shp"

# Load the shapefiles
forest_depth_data = gpd.read_file(forest_depth_path)
summed_edge_length_data = gpd.read_file(summed_edge_length_path)

# Display the first few rows of the data
print("Forest Depth Data:")
print(forest_depth_data.head())

print("\nSummed Edge Length Data:")
print(summed_edge_length_data.head())

In [ ]:
import geopandas as gpd
import pandas as pd

# 加载数据
forest_depth_data = gpd.read_file("G:/Hangkai/CONUS Forest Edge Mapping/2019/CONUS_2019_Forest_Depth.shp")
summed_edge_length_data = gpd.read_file("G:/Hangkai/CONUS Forest Edge Mapping/2019/CONUS_2019_summed_edge_length.shp")

# 对Forest Depth Data进行统计
forest_columns = ['Forest_D_1', 'Forest_D_2', 'Forest_D_3', 'Forest_D_4', 'Forest_D_5']
depth_counts = forest_depth_data.groupby('NA_L1KEY')[forest_columns].sum()

print("Forest Depth Counts by Ecoregion:")
print(depth_counts)

# 更新列名生成方式：1到9是summed_e1到summed_e9，之后是summed_e10到summed_e42
edge_columns = ['summed_e_' + str(i) for i in range(1, 10)] + ['summed_e' + str(i) for i in range(10, 42)]
edge_sum = summed_edge_length_data.groupby('NA_L1KEY')[edge_columns].sum()

print("\nSummed Edge Length Averages by Ecoregion:")
print(edge_sum)

# 可选：保存统计结果到CSV文件
depth_counts.to_csv("forest_depth_counts.csv")
edge_sum.to_csv("summed_edge_length_sum.csv")


In [ ]:
import pandas as pd

# 假设depth_counts已经加载
# depth_counts = pd.read_csv("forest_depth_counts.csv")

# 计算总森林面积（以平方千米为单位）
total_forest_area_km2 = depth_counts.sum().sum() * 900 / 1e6

# 不同深度的森林面积及其比例
depth_total = depth_counts.sum()
depth_proportion = depth_total / depth_total.sum()

# 每个ecoregion的森林面积及其比例
ecoregion_area = depth_counts.sum(axis=1)
total_area = ecoregion_area.sum()
ecoregion_proportion = ecoregion_area / total_area

# 不同ecoregion不同深度的森林面积比例
ecoregion_depth_proportion = depth_counts.div(depth_counts.sum(axis=1), axis=0)

# 输出结果
print("总森林面积（平方千米）:", total_forest_area_km2)
print("\n不同深度的森林面积及其比例:\n", depth_proportion)
print("\n每个ecoregion的森林面积及其比例:\n", ecoregion_proportion)
print("\n不同ecoregion不同深度的森林面积比例:\n", ecoregion_depth_proportion)


In [ ]:
import matplotlib.pyplot as plt

# 不同深度的森林面积比例数据
depth_proportions = {
    '0-30m forest': 0.263520,
    '30-60m forest': 0.156717,
    '60-90m forest': 0.110652,
    '90-120m forest': 0.083488,
    '>120m forest': 0.385623
}
# 设置字体大小
plt.rcParams['font.size'] = 30  # 调整这里以改变标签和百分比的字体大小
plt.rcParams['axes.titlesize'] = 20  # 调整这里以改变标题的字体大小
# 创建饼图
plt.figure(figsize=(8, 8))
plt.pie(depth_proportions.values(), labels=depth_proportions.keys(), autopct='%1.1f%%', colors=['#ff9999','#66b3ff','#99ff99','#ffcc99', '#c2c2f0'])
plt.title('Proportion of Forest Depths Across the US')
plt.show()


In [ ]:
summed_edge_length_data

In [ ]:
import matplotlib.pyplot as plt

# 不同ecoregion的森林面积比例数据
ecoregion_proportions = {
    'WATER': 0.000068,
    'NORTH AMERICAN DESERTS': 0.028350,
    'MEDITERRANEAN CALIFORNIA': 0.006831,
    'SOUTHERN SEMIARID HIGHLANDS': 0.001082,
    'TEMPERATE SIERRAS': 0.019629,
    'TROPICAL WET FORESTS': 0.002878,
    'NORTHERN FORESTS': 0.121337,
    'NORTHWESTERN FORESTED MOUNTAINS': 0.188710,
    'MARINE WEST COAST FOREST': 0.025844,
    'EASTERN TEMPERATE FORESTS': 0.557216,
    'GREAT PLAINS': 0.048055
}

# 设置字体大小
plt.rcParams['font.size'] = 18
plt.rcParams['axes.titlesize'] = 20

# 创建饼图
fig, ax = plt.subplots(figsize=(12, 12))  # 增加图表大小
wedges, texts, autotexts = ax.pie(ecoregion_proportions.values(), 
                                  autopct='%1.1f%%', colors=plt.cm.Paired.colors)

# 使用图例
ax.legend(wedges, ecoregion_proportions.keys(),
          title="Ecoregions",
          loc="center left",
          bbox_to_anchor=(1, 0, 0.5, 1))

plt.setp(autotexts, size=20)  # 设置百分比标签的大小
plt.title('Forest Area Proportion by Ecoregion')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 不同ecoregion的森林面积比例数据
ecoregion_proportions = {
    'WATER': 0.000068,
    'NORTH AMERICAN DESERTS': 0.028350,
    'MEDITERRANEAN CALIFORNIA': 0.006831,
    'SOUTHERN SEMIARID HIGHLANDS': 0.001082,
    'TEMPERATE SIERRAS': 0.019629,
    'TROPICAL WET FORESTS': 0.002878,
    'NORTHERN FORESTS': 0.121337,
    'NORTHWESTERN FORESTED MOUNTAINS': 0.188710,
    'MARINE WEST COAST FOREST': 0.025844,
    'EASTERN TEMPERATE FORESTS': 0.557216,
    'GREAT PLAINS': 0.048055
}

# 设置字体大小和图表大小
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 18

# 创建饼图
fig, ax = plt.subplots(figsize=(10, 8))  # 调整图表大小为合适的尺寸
wedges, texts, autotexts = ax.pie(ecoregion_proportions.values(), colors=plt.cm.Paired.colors)

# 生成标签列表包含百分比
labels = [f'{key}: {value*100:.2f}%' for key, value in ecoregion_proportions.items()]

# 在图例中显示百分比，图例放置于图表旁边
ax.legend(wedges, labels, title="Ecoregions", loc="center left", bbox_to_anchor=(1, 0.5))

plt.title('Forest Area Proportion by Ecoregion')
plt.tight_layout()  # 调整整体布局以防止切割
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 示例数据，假设这是之前的数据
data = {
    '0-30m forest': [0.599383, 0.505428, 0.570854, 0.527360, 0.296337, 0.423280, 0.137112, 0.245827, 0.148448, 0.252644, 0.624397],
    '30-60m forest': [0.136337, 0.204201, 0.193047, 0.209750, 0.183951, 0.165232, 0.115441, 0.155900, 0.104677, 0.161301, 0.193003],
    '60-90m forest': [0.080329, 0.106334, 0.091798, 0.104471, 0.125662, 0.095416, 0.100164, 0.111437, 0.084044, 0.116322, 0.082799],
    '90-120m forest': [0.052907, 0.061789, 0.050769, 0.057625, 0.090339, 0.063439, 0.087201, 0.083820, 0.070528, 0.088224, 0.041346],
    '>120m forest': [0.131044, 0.122247, 0.093531, 0.100793, 0.303711, 0.252632, 0.560083, 0.403016, 0.592303, 0.381510, 0.058455]
}

index = [
    'WATER',
    'NORTH AMERICAN DESERTS',
    'MEDITERRANEAN CALIFORNIA',
    'SOUTHERN SEMIARID HIGHLANDS',
    'TEMPERATE SIERRAS',
    'TROPICAL WET FORESTS',
    'NORTHERN FORESTS',
    'NORTHWESTERN FORESTED MOUNTAINS',
    'MARINE WEST COAST FOREST',
    'EASTERN TEMPERATE FORESTS',
    'GREAT PLAINS'
]

df = pd.DataFrame(data, index=index)
df = df.sort_values('>120m forest', ascending=False)

# 创建堆叠条形图
ax = df.plot(kind='bar', stacked=True, figsize=(14, 8), colormap='viridis')
ax.set_title('Proportion of Forest Depths by Ecoregion')
ax.set_ylabel('Proportion')
plt.xticks(rotation=30,ha='right')
ax.legend(title='Forest Depth Categories', loc='center left', bbox_to_anchor=(1, 0.5))
#plt.tight_layout()
plt.show()

In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 模拟之前的代码部分，假设edge_sums已经正确计算
# edge_columns = ['summed_e' + ('_' if i < 10 else '') + str(i) for i in range(1, 43)]
# edge_sums = summed_edge_length_data.groupby('NA_L1KEY')[edge_columns].sum()

df = []
# 创建DataFrame
df = pd.DataFrame(edge_sum)

# 汇总每个edge长度的影响
total_edge_impact = df.sum()

# 将像素数转换为面积（平方公里）
total_edge_impact_km2 = total_edge_impact * 900 / 1e6  # 900平方米转为平方公里

# 创建条形图展示每种edge长度的影响
plt.figure(figsize=(14, 8))
plt.bar(range(1, 42), total_edge_impact_km2, color='teal')
plt.title('Total Forest Edge Area Affected by Different Edge Lengths Across the CONUS (in km²)')
plt.xlabel('Edge Length (*30 m)')
plt.ylabel('Area Affected (km²)')
plt.xticks(range(1, 43), rotation=90)  # 设置x轴标签为1到42
plt.grid(True)
plt.show()

In [ ]:
df_eco = df
df

In [ ]:
import numpy as np
edge_lengths = np.arange(1, 42)
total_edge_length_meters = (total_edge_impact * edge_lengths).sum()*30

# 计算总受影响的面积（平方米）
total_affected_area_m2 = df.sum(axis=1) * 900

# 计算平均影响（每平方米受到的边缘长度）
average_edge_per_m2 = total_edge_length_meters / total_affected_area_m2

print("Total edge length affected (kilometers):", total_edge_length_meters)
print("Total affected area (square kilometers):", total_affected_area_m2)
print("Average edge impact per square kilometers:", average_edge_per_m2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data provided in the question
data = {
    'Ecoregion': [
        "NORTH AMERICAN DESERTS", "MEDITERRANEAN CALIFORNIA",
        "SOUTHERN SEMIARID HIGHLANDS", "TEMPERATE SIERRAS", "TROPICAL WET FORESTS",
        "NORTHERN FORESTS", "NORTHWESTERN FORESTED MOUNTAINS", "MARINE WEST COAST FOREST",
        "EASTERN TEMPERATE FORESTS", "GREAT PLAINS"
    ],
    'Average Edge Impact per sq km': [
        5.990304e+03, 2.361025e+04, 1.520175e+05, 1.180550e+04,
        7.052132e+04, 3.333839e+03, 1.453367e+03, 1.629302e+04, 4.769996e+02,
        3.179649e+03
    ]
}

# Create DataFrame
df1 = pd.DataFrame(data)

# Sorting the data for better visualization
df1 = df1.sort_values('Average Edge Impact per sq km', ascending=False)

### Step 2: Visualization

plt.figure(figsize=(12, 8))
plt.bar(df1['Ecoregion'], df1['Average Edge Impact per sq km'], color='cadetblue')
plt.title('Average Edge Impact per Square Kilometer by Ecoregion')
plt.xlabel('Ecoregion')
plt.ylabel('Average Edge Impact (km/km²)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()  # Adjust layout to make room for the rotated x-axis labels

plt.show()


In [ ]:
# 假设total_edge_impact包含了每种边缘长度的受影响像素总数
total_edge_impact = df.sum()  # 重新确认数据已经按照之前的步骤汇总

# 计算总受影响的边缘长度（米）
# 使用numpy的arange生成边缘长度数组：1到42
import numpy as np
edge_lengths = np.arange(1, 42)
total_edge_length_meters = (total_edge_impact * edge_lengths).sum()*30/10e3

# 计算总受影响的面积（平方米）
total_affected_area_m2 = total_edge_impact.sum() * 900/10e6

# 计算平均影响（每平方米受到的边缘长度）
average_edge_per_m2 = total_edge_length_meters / total_affected_area_m2

print("Total edge length affected (kilometers):", total_edge_length_meters)
print("Total affected area (square kilometers):", total_affected_area_m2)
print("Average edge impact per square kilometers:", average_edge_per_m2)

In [ ]:
# 计算总受影响的边缘长度（米）
# 使用numpy的arange生成边缘长度数组：1到42
import numpy as np
edge_lengths = np.arange(1, 42)
eco_total_edge_length_meters = (df_eco * edge_lengths).sum()*30/10e3

# 计算总受影响的面积（平方米）
eco_total_affected_area_m2 = eco_total_edge_length_meters.sum() * 900/10e6

# 计算平均影响（每平方米受到的边缘长度）
average_edge_per_m2 = total_edge_length_meters / eco_total_affected_area_m2

print("Total edge length affected (kilometers):", total_edge_length_meters)
print("Total affected area (square kilometers):", total_affected_area_m2)
print("Average edge impact per square kilometers:", average_edge_per_m2)